In [1]:
import os
import ast
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
all_contig_data = []
cog_data = []

for genus_name in keep_genus:
    target_folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
    contig_info = pd.read_csv(f'{target_folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    contig_info['genus'] = genus_name
    rep_cog_data = pd.read_csv(f'{target_folder}/COG_statistics.tsv', sep='\t')
    
    all_contig_data.append(contig_info)
    cog_data.append(rep_cog_data)

all_contig_data = pd.concat(all_contig_data, ignore_index=True)
all_contig_data = all_contig_data[['accession', 'organism', 'genus', 'category-pident_90', 'size']]

cog_data = pd.concat(cog_data, ignore_index=True)
cog_data = pd.concat([cog_data,cog_data.loc[:,cog_data.columns[6:]].div(cog_data["All coding CDSs"],axis=0).add_suffix("_cog_freq")],axis=1)
cog_data_freq = cog_data[['accession'] + [k for k in cog_data.columns if 'cog_freq' in k]]

all_contig_data = pd.merge(all_contig_data, cog_data_freq, on='accession', how='left')
all_contig_data = all_contig_data.dropna(axis=0, how='any')

In [3]:
from tqdm import tqdm

def load_single_kmer(kmer_dir, record_id, target_k):
    kmer_file = os.path.join(kmer_dir, f"{record_id}.txt")
    if not os.path.exists(kmer_file):
        return None
    with open(kmer_file, 'r') as f:
        raw = f.read()
        kmer_dict = ast.literal_eval(raw)
        
    k_key = f"{target_k}-mer"
    if k_key not in kmer_dict:
        return None
    kmer_counts = kmer_dict[k_key]
    valid_bases = {"A", "T", "C", "G"}
    clean_counts = {}
    for seq, cnt in kmer_counts.items():
        if set(seq).issubset(valid_bases):
            clean_counts[seq] = cnt
    total = sum(clean_counts.values())
    if total == 0:
        return None
    return {f"{seq}_{target_k}mer_freq": cnt / total for seq, cnt in clean_counts.items()}

kmer_base_dir = "/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/kmer"
kmer_data = []
with tqdm(total = len(all_contig_data), desc=f'k-mer_COG', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
    for idx, row in all_contig_data.iterrows():
        acc_n, record_id = row["accession"].split('-')
        kmer_dir = os.path.join(kmer_base_dir, acc_n)
        kmer_freq = {}
        for k in [3, 4, 5]:
            kmer_freq = kmer_freq | load_single_kmer(kmer_dir, record_id, k)
        if kmer_freq is None:
            pbar.update(1)
            continue
        kmer_freq["accession"] = row["accession"]
        kmer_data.append(kmer_freq)
        pbar.update(1)

kmer_df = pd.DataFrame(kmer_data).fillna(0)

k-mer_COG: 100%|████████████████████████████████████████████████| 72.0k/72.0k [17:37<00:00, 68.1B/s]


In [4]:
all_contig_data = pd.merge(all_contig_data, kmer_df, on='accession', how='left')
all_contig_data

,accession,organism,genus,category-pident_90,size,L_cog_freq,B_cog_freq,D_cog_freq,Y_cog_freq,V_cog_freq,...,AAGGG_5mer_freq,GGGGG_5mer_freq,TCTAG_5mer_freq,GACCG_5mer_freq,CTTGG_5mer_freq,CTAGC_5mer_freq,CCTAG_5mer_freq,CTAGA_5mer_freq,GCTAG_5mer_freq,CTAGG_5mer_freq
0,GCF_000512125.1-NZ_CP007025.1,Escherichia albertii,Escherichia,typical chromosome,4701875,0.032212,0.001670,0.013362,0.0,0.023861,...,0.000515,0.000365,0.000035,0.000834,0.000266,0.000081,0.000030,0.000035,0.000081,0.000030
1,GCF_000597845.1-NZ_CP007265.1,Escherichia coli,Escherichia,typical chromosome,4758629,0.027618,0.001381,0.011047,0.0,0.023705,...,0.000547,0.000320,0.000032,0.000927,0.000256,0.000077,0.000028,0.000032,0.000077,0.000028
2,GCF_000599625.1-NZ_CP007390.1,Escherichia coli,Escherichia,typical chromosome,4807977,0.014158,0.000457,0.004796,0.0,0.011418,...,0.000550,0.000324,0.000033,0.000921,0.000260,0.000079,0.000030,0.000033,0.000079,0.000030
3,GCF_000599645.1-NZ_CP007391.1,Escherichia coli,Escherichia,typical chromosome,4875682,0.014512,0.000454,0.005442,0.0,0.011338,...,0.000546,0.000324,0.000034,0.000920,0.000259,0.000079,0.000030,0.000034,0.000079,0.000030
4,GCF_000599665.1-NZ_CP007392.1,Escherichia coli,Escherichia,typical chromosome,5054509,0.020897,0.001320,0.008799,0.0,0.018478,...,0.000551,0.000343,0.000035,0.000934,0.000250,0.000079,0.000032,0.000035,0.000079,0.000032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72031,GCF_900176185.1-NZ_LT838273.1,Helicobacter pylori,Helicobacter,typical chromosome,1682800,0.061737,0.001357,0.023745,0.0,0.043419,...,0.001805,0.001215,0.000751,0.000160,0.000872,0.000909,0.000551,0.000751,0.000909,0.000551
72032,GCF_900478295.1-NZ_LS483488.1,Helicobacter pylori,Helicobacter,typical chromosome,1680937,0.069081,0.001341,0.023474,0.0,0.039571,...,0.001737,0.001138,0.000744,0.000144,0.000871,0.000901,0.000550,0.000744,0.000901,0.000550
72033,GCF_900638475.1-NZ_LR134517.1,Helicobacter pylori,Helicobacter,typical chromosome,1644315,0.065007,0.001383,0.023513,0.0,0.038728,...,0.001765,0.001189,0.000735,0.000161,0.000872,0.000911,0.000548,0.000735,0.000911,0.000548
72034,GCF_900638505.1-NZ_LR134519.1,Helicobacter pylori,Helicobacter,typical chromosome,1632224,0.061484,0.001413,0.021908,0.0,0.039576,...,0.001748,0.001156,0.000745,0.000147,0.000859,0.000909,0.000550,0.000745,0.000909,0.000550


In [5]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                               roc_auc_score, roc_curve, average_precision_score, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

BASE_ROOT = "/active-data/analysis_results/chr_pla/genus/machine_learning"
k_list = [3, 4, 5]
size_switch_list = [True, False]

df_full = all_contig_data.copy()

for k in k_list:
    for use_size in size_switch_list:
        tag_size = "with_size" if use_size else "no_size"
        exp_dir = os.path.join(BASE_ROOT, f"k{k}_{tag_size}")
        os.makedirs(exp_dir, exist_ok=True)
        print("=" * 60)
        print(f"Running experiment: k={k}, {tag_size}, output dir: {exp_dir}")
        print("=" * 60)

        kmer_suffix = f"{k}mer_freq"
        kmer_cols = [c for c in df_full.columns if c.endswith(kmer_suffix)]
        cog_cols = [c for c in df_full.columns if c.endswith("cog_freq")]
        feature_cols = kmer_cols + cog_cols

        if use_size:
            feature_cols = ["size"] + feature_cols
        print(f"Total feature count: {len(feature_cols)}")

        X = df_full[feature_cols].values
        y_raw = df_full["category-pident_90"].values

        le = LabelEncoder()
        y = le.fit_transform(y_raw)
        print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )

        lr_pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(multi_class="multinomial", max_iter=3000, class_weight="balanced", n_jobs=-1))
        ])
        rf_pipe = Pipeline([
            ("clf", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced"))
        ])
        linear_svm_pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(class_weight="balanced", max_iter=8000, multi_class="ovr"))
        ])

        def run_model(model, name):
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            print(f"\n==== {name} Result ====")
            print(f"Test Accuracy: {acc:.4f}")
            report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
            print(classification_report(y_test, y_pred, target_names=le.classes_))
            cm = confusion_matrix(y_test, y_pred)
            print("Confusion Matrix:")
            print(cm)
            pd.DataFrame(report).T.to_csv(os.path.join(exp_dir, f"{name}_report.csv"))
            pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_csv(os.path.join(exp_dir, f"{name}_confusion_matrix.csv"))

            n_classes = len(le.classes_)
            y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

            if name == "Linear SVM":
                y_score = model.decision_function(X_test)
            else:
                y_score = model.predict_proba(X_test)

            auc_result = []
            ap_result = []
            roc_curve_list = []
            pr_curve_list = []

            for idx, cls in enumerate(le.classes_):
                y_true_ovr = y_test_bin[:, idx]
                ys = y_score[:, idx]

                try:
                    ovr_auc = roc_auc_score(y_true_ovr, ys)
                except ValueError:
                    ovr_auc = np.nan
                auc_result.append({"class": cls, "ovr_auc": ovr_auc})

                try:
                    ovr_ap = average_precision_score(y_true_ovr, ys)
                except ValueError:
                    ovr_ap = np.nan
                ap_result.append({"class": cls, "ovr_ap": ovr_ap})

                fpr, tpr, thr_roc = roc_curve(y_true_ovr, ys)
                roc_curve_list.append(pd.DataFrame({
                    "class": cls,
                    "fpr": fpr,
                    "tpr": tpr,
                    "threshold": thr_roc
                }))

                precision, recall, thr_pr = precision_recall_curve(y_true_ovr, ys)
                pr_curve_list.append(pd.DataFrame({
                    "class": cls,
                    "precision": precision,
                    "recall": recall
                }))

            pd.DataFrame(auc_result).to_csv(os.path.join(exp_dir, f"{name}_ovr_auc.csv"), index=False)
            pd.DataFrame(ap_result).to_csv(os.path.join(exp_dir, f"{name}_ovr_ap.csv"), index=False)
            pd.concat(roc_curve_list).to_csv(os.path.join(exp_dir, f"{name}_ovr_roc_curve.csv"), index=False)
            pd.concat(pr_curve_list).to_csv(os.path.join(exp_dir, f"{name}_ovr_pr_curve.csv"), index=False)

            return model

        rf = run_model(rf_pipe, "Random Forest")
        lr = run_model(lr_pipe, "Logistic Regression")
        svm = run_model(linear_svm_pipe, "Linear SVM")

        rf_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": rf.named_steps["clf"].feature_importances_
        }).sort_values("importance", ascending=False)
        rf_imp.to_csv(os.path.join(exp_dir, "rf_feature_importance.csv"), index=False)
        print("\nTop20 Important Features:")
        print(rf_imp.head(20))

        lr_weight = pd.DataFrame(
            lr.named_steps["clf"].coef_.T,
            columns=le.classes_,
            index=feature_cols
        )
        lr_weight.to_csv(os.path.join(exp_dir, "lr_feature_coef.csv"))

        svm_weight = pd.DataFrame(
            svm.named_steps["clf"].coef_.T,
            columns=le.classes_,
            index=feature_cols
        )
        svm_weight.to_csv(os.path.join(exp_dir, "linear_svm_feature_coef.csv"))

        joblib.dump(rf, os.path.join(exp_dir, "contig_3class_rf.pkl"))
        joblib.dump(lr, os.path.join(exp_dir, "contig_3class_lr.pkl"))
        joblib.dump(svm, os.path.join(exp_dir, "contig_3class_linearsvm.pkl"))
        joblib.dump(le, os.path.join(exp_dir, "label_encoder.pkl"))

print("\nAll experiments finished!")

Running experiment: k=3, with_size, output dir: /active-data/analysis_results/chr_pla/genus/machine_learning/k3_with_size
Total feature count: 88
Class mapping: {'intermediate replicon': np.int64(0), 'typical chromosome': np.int64(1), 'typical plasmid': np.int64(2)}

==== Random Forest Result ====
Test Accuracy: 0.9904
                       precision    recall  f1-score   support

intermediate replicon       0.91      0.28      0.43       268
   typical chromosome       1.00      1.00      1.00      9003
      typical plasmid       0.98      1.00      0.99     12340

             accuracy                           0.99     21611
            macro avg       0.97      0.76      0.81     21611
         weighted avg       0.99      0.99      0.99     21611

Confusion Matrix:
[[   75     0   193]
 [    0  9002     1]
 [    7     6 12327]]

==== Logistic Regression Result ====
Test Accuracy: 0.9175
                       precision    recall  f1-score   support

intermediate replicon       0